# Comparing Ansible execution patterns: block/rescue/always, serial rollout, and max_fail_percentage

last_verified: 2026-09-17 · Ansible 14.4.0 / ansible-core 2.21.4

## Purpose

These three settings answer three different questions about failure. `block`/`rescue`/`always` handles failure *within one host's task list*: try the primary step, fall back when it fails, and always run cleanup. `serial` controls *how many hosts run at once*, so a rolling update never takes the whole fleet down together. `max_fail_percentage` decides *when the entire play stops*, aborting a rollout that is failing too broadly to continue. Mixing them up produces either fragile automation (no recovery path) or dangerous automation (a bad change pushed everywhere at full speed). This notebook runs all three against local demo hosts so the behavioral differences are visible in real output.

## When to use which

| Pattern | Scope | Reach for it when |
|---|---|---|
| `block` / `rescue` / `always` | tasks on a single host | one step may fail but the host can recover: retry with a fallback, restore a previous version, remove a temp file |
| `serial` | hosts in a play | capacity must stay up during the change: rolling web deploy, node-by-node upgrade |
| `max_fail_percentage` | the whole play | some bad hosts are tolerable but a widespread failure means the change itself is broken: abort instead of finishing |

The three compose: a rolling (`serial`) deploy where each host has a `block`/`rescue` fallback, guarded by a `max_fail_percentage` abort threshold, is the standard shape for a safe fleet-wide change.

## Prerequisites

- `ansible-playbook` on PATH (examples written against Ansible 14.4.0 with ansible-core 2.21.4).
- No remote hosts needed: the demo inventory maps four logical hosts onto the local machine.
- One note from the Ansible 14 porting guidance that shapes these examples: task failure should be explicit (the `fail` module, `failed_when`) rather than inferred from a bare non-zero return code, so the failing steps below say exactly what they mean.

In [ ]:
import os
import shutil
import subprocess
import tempfile
import textwrap

tmp = tempfile.mkdtemp(prefix="ansible_exec_patterns_")
print(f"Working in {tmp}")

inventory = textwrap.dedent("""\
    [demo]
    web1 ansible_host=127.0.0.1 ansible_connection=local
    web2 ansible_host=127.0.0.1 ansible_connection=local
    web3 ansible_host=127.0.0.1 ansible_connection=local
    web4 ansible_host=127.0.0.1 ansible_connection=local
    """)
with open(os.path.join(tmp, "inventory.ini"), "w") as f:
    f.write(inventory)

HAS_ANSIBLE = shutil.which("ansible-playbook") is not None
print("ansible-playbook available:", HAS_ANSIBLE)


def run_play(playbook_name, extra_args=None):
    """Run a playbook from the temp dir; return the result or None."""
    if not HAS_ANSIBLE:
        print(f"SKIP {playbook_name}: ansible-playbook not on PATH")
        return None
    try:
        return subprocess.run(
            ["ansible-playbook", "-i", os.path.join(tmp, "inventory.ini"),
             os.path.join(tmp, playbook_name)] + (extra_args or []),
            capture_output=True, text=True, timeout=120,
        )
    except subprocess.TimeoutExpired as exc:
        print(f"TIMEOUT {playbook_name}: {exc}")
        return None

## Pattern 1 — block/rescue/always (recover on one host)

The `block` groups the primary steps. If any step in the block fails, execution jumps to `rescue` instead of failing the host. `always` runs in both cases, which makes it the right place for cleanup or status reporting. Here the primary step fails on purpose, `rescue` records a fallback version, and `always` announces that the attempt finished.

In [ ]:
block_play = textwrap.dedent("""\
    - name: Handle a recoverable failure with block/rescue/always
      hosts: web1
      gather_facts: false
      tasks:
        - name: Attempt the primary step
          block:
            - name: Primary step that fails on purpose
              ansible.builtin.command: /bin/false
          rescue:
            - name: Fall back to a safe default
              ansible.builtin.set_fact:
                deployed_version: "previous-stable"
            - name: Report the recovery
              ansible.builtin.debug:
                msg: "Primary step failed; rescued with {{ deployed_version }}"
          always:
            - name: Record that the attempt finished
              ansible.builtin.debug:
                msg: "Attempt finished (success or rescue)"
    """)
with open(os.path.join(tmp, "01-block-rescue.yml"), "w") as f:
    f.write(block_play)

result = run_play("01-block-rescue.yml")
if result is not None:
    print(result.stdout[-1500:])
    print("returncode:", result.returncode)

Expected: the play succeeds (`returncode` 0) even though the primary step failed, because `rescue` handled it. The output shows both the rescue message and the `always` message. Without the `rescue` section, the same failure would fail the host and stop its task list.

## Pattern 2 — serial (roll through the fleet in batches)

`serial: 2` runs the play against two hosts at a time: web1+web2 first, then web3+web4. A failure in an early batch still blocks later batches by default, which is what makes this a rollout rather than a fan-out. The `ansible_play_batch` variable exposes which hosts are in the current batch, useful for draining a host from a load balancer before touching it.

In [ ]:
serial_play = textwrap.dedent("""\
    - name: Rolling update two hosts at a time
      hosts: demo
      gather_facts: false
      serial: 2
      tasks:
        - name: Update one host out of its batch
          ansible.builtin.debug:
            msg: "Updating {{ inventory_hostname }} (batch: {{ ansible_play_batch }})"
    """)
with open(os.path.join(tmp, "02-serial.yml"), "w") as f:
    f.write(serial_play)

result = run_play("02-serial.yml")
if result is not None:
    print(result.stdout[-2000:])
    print("returncode:", result.returncode)

Expected: the PLAY header appears twice (one per batch of two), and each host's message names its batch mates. Contrast with the default behavior — no `serial` — where all four hosts run in a single batch under one PLAY header.

## Pattern 3 — max_fail_percentage (abort a broken rollout)

`max_fail_percentage: 25` lets the play continue while failures stay at or below a quarter of the hosts, and aborts the rest once failures exceed that share. Here two of four hosts fail on purpose (50%), so the play must stop early instead of finishing the healthy hosts' remaining work. The failing step uses the `fail` module so the failure is explicit and carries a useful message.

In [ ]:
pct_play = textwrap.dedent("""\
    - name: Abort the rollout when too many hosts fail
      hosts: demo
      gather_facts: false
      max_fail_percentage: 25
      tasks:
        - name: Fail on half the fleet on purpose
          ansible.builtin.fail:
            msg: "Simulated bad deploy on {{ inventory_hostname }}"
          when: inventory_hostname in ["web3", "web4"]
        - name: Only hosts that passed reach this step
          ansible.builtin.debug:
            msg: "{{ inventory_hostname }} is healthy"
    """)
with open(os.path.join(tmp, "03-max-fail-pct.yml"), "w") as f:
    f.write(pct_play)

result = run_play("03-max-fail-pct.yml")
if result is not None:
    print(result.stdout[-2000:])
    print("returncode:", result.returncode, "(non-zero expected: the rollout aborted)")

## Verify

- Pattern 1: `returncode` is 0 and the output contains both `rescued with previous-stable` and `Attempt finished`. Re-run with the `rescue` section deleted to confirm the same play then fails — that contrast is the whole point of the pattern.
- Pattern 2: the PLAY header repeats once per batch, and each host's `batch:` list holds exactly its batch mate. Change `serial: 2` to `serial: 1` and confirm four PLAY headers appear.
- Pattern 3: `returncode` is non-zero, the recap shows failures on web3/web4, and the healthy-hosts step never reports for the hosts that never ran. Lower the failure to one host (25%, at the threshold) and confirm the play completes instead of aborting.

## Common errors

- Over-wide `block`: putting ten tasks in one block means any of them can trigger the same `rescue`, hiding which step actually broke. Keep blocks small so the rescue matches the failure it was written for.
- `rescue` that can fail too: if a rescue task fails, the host fails despite the handler. Guard rescue steps (or add a second-level fallback) when they touch anything unreliable.
- `serial` percentage surprises: `serial: "50%"` is computed per batch against the remaining hosts, so small fleets still move one host at a time. Prefer an absolute number for small inventories.
- `max_fail_percentage` measured against the wrong denominator: the percentage counts hosts in the current play (or the current `serial` batch), not the whole inventory. A 25% threshold on a two-host batch aborts after a single failure.
- Expecting `always` to mean success: `always` runs even when `rescue` also failed. Check the play recap, not the presence of `always` output, to judge the outcome.

## References

- Ansible 14 porting guide (explicit-failure guidance, collection compatibility): https://docs.ansible.com/projects/ansible/latest/porting_guides/porting_guide_14.html
- Version context: Ansible 14.4.0 (https://pypi.org/pypi/ansible/json) with ansible-core 2.21.4 (https://pypi.org/pypi/ansible-core/json).
- The three playbooks above are self-contained in this notebook's temp dir (`01-block-rescue.yml`, `02-serial.yml`, `03-max-fail-pct.yml`); re-run any cell to reproduce its output.